# Train "Hey IRIS" wake word model

This notebook trains a custom openWakeWord model for the phrase **"Hey IRIS"**.

Click **Runtime → Run all** and wait ~75–90 minutes (T4 GPU) or ~2.5 hours (free CPU).

When done, `hey_iris.onnx` will download automatically. Place it in your Android project at:
`app/src/main/assets/hey_iris.onnx`

In [ ]:
#@title 1. Install dependencies
!pip install -q openwakeword torch torchaudio speechbrain datasets scipy tqdm 2>&1 | tail -3

In [ ]:
#@title 2. Define training config
TARGET_PHRASE = ['hey iris']   #@param {type:"string"}
MODEL_NAME    = 'hey_iris'      #@param {type:"string"}
N_SAMPLES     = 10000           #@param {type:"integer"}
TRAINING_STEPS = 15000          #@param {type:"integer"}
LAYER_SIZE    = 32              #@param {type:"integer"}

print(f"Target: {TARGET_PHRASE}")
print(f"Model: {MODEL_NAME}")
print(f"Samples: {N_SAMPLES}, Steps: {TRAINING_STEPS}, Layer: {LAYER_SIZE}")

In [ ]:
#@title 3. Mount Google Drive (optional — saves model here too)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted at /content/drive")

In [ ]:
#@title 4. Generate synthetic "Hey IRIS" samples
import os
import torchaudio
import torch
import numpy as np
from tqdm import tqdm
import math
import scipy.signal as signal

output_dir = f"my_custom_model/{MODEL_NAME}"
os.makedirs(f"{output_dir}/positive", exist_ok=True)
os.makedirs(f"{output_dir}/negative", exist_ok=True)

# Use Piper TTS via the openwakeword data generation utils
from openwakeword.synthetic_data import generate_synthetic_audio

print("Generating positive samples...")
generate_synthetic_audio(
    phrases=TARGET_PHRASE,
    output_dir=f"{output_dir}/positive",
    n_samples=N_SAMPLES,
    sample_rate=16000
)

print(f"Generated {len(os.listdir(f'{output_dir}/positive'))} positive samples")
print("Done")

In [ ]:
#@title 5. Prepare training data & train model
from openwakeword.train import train_model

train_model(
    model_name=MODEL_NAME,
    positive_dir=f"{output_dir}/positive",
    output_dir=output_dir,
    steps=TRAINING_STEPS,
    layer_size=LAYER_SIZE,
    batch_size=32,
    learning_rate=0.001
)

print("Training complete!")

In [ ]:
#@title 6. Download model
from google.colab import files
import os

model_path = f"{output_dir}/{MODEL_NAME}.onnx"
if os.path.exists(model_path):
    files.download(model_path)
    print(f"Model downloaded: {model_path}")
else:
    print(f"Model not found at {model_path}")
    print("Files in output dir:", os.listdir(output_dir))

In [ ]:
#@title 7. (Optional) Copy to Google Drive
import shutil
drive_path = f"/content/drive/MyDrive/iris_wakeword/{MODEL_NAME}.onnx"
os.makedirs("/content/drive/MyDrive/iris_wakeword", exist_ok=True)
if os.path.exists(model_path):
    shutil.copy(model_path, drive_path)
    print(f"Copied to Drive: {drive_path}")
else:
    print("Model not found, skipping Drive copy")